# 08 | Common Forecasting Baselines (2025 Validation)
**NHS A&E Intelligent Decision Support System**

**Objective:** evaluate two deterministic, publication-aware forecasting baselines on the approved 2025 validation population (`split == "validation"`, `evaluation_eligible == True`, `1,452` observations):
1. **Latest-published attendance baseline (`T - 2`):** `pred = type1_origin`
2. **Seasonal naive baseline (`T - 12`):** `pred = type1_target_lag_12`

All calculations reuse `src/models/baselines.py`, `src/models/evaluation_splits.py`, and `src/models/feature_engineering.py`. The 2026 final test partition is strictly unexamined in this notebook.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

project_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents) if (p / "docs/model_input_manifest.json").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Open the project repository before running this notebook.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models.baselines import evaluate_validation_baselines
from src.models.evaluation_splits import assign_chronological_splits, load_verified_features

feat_df, provenance = load_verified_features(project_root)
split_df = assign_chronological_splits(feat_df)
results = evaluate_validation_baselines(split_df)

summary = results["summary"]
predictions_df = results["predictions_df"]
comparison_df = results["comparison_df"]
monthly_df = results["monthly_breakdown_df"]

print("M06 run ID:", provenance["m06_run_id"])
print("M06 SHA-256:", provenance["m06_csv_sha256"])
print("Feature CSV SHA-256:", provenance["feature_csv_sha256"])
print(
    f"Validation scored: {summary['scored_observations']:,} observations | "
    f"{summary['scored_distinct_providers']} providers | "
    f"{summary['target_period_start']} to {summary['target_period_end']} | "
    f"Absent targets excluded: {summary['excluded_absent_target_rows']} ({summary['excluded_absent_target_codes']}) | "
    f"Test evaluated: {summary['test_set_evaluated']}"
)


## 1. Manual Verification Example (`R0A`, Target October 2025)
Cross-check the baseline predictions for Manchester University NHS Foundation Trust (`R0A`) targeting October 2025 directly against the verified Step 5 feature dataset.

In [ ]:
r0a_pred = predictions_df.loc[
    (predictions_df["org_code"] == "R0A") & (predictions_df["target_period"] == pd.Timestamp("2025-10-01"))
].iloc[0]
r0a_src = feat_df.loc[
    (feat_df["org_code"] == "R0A") & (feat_df["target_period"] == pd.Timestamp("2025-10-01"))
].iloc[0]

verify_table = pd.DataFrame([
    ("Provider", r0a_pred["org_code"], r0a_src["org_code"]),
    ("Forecast decision date", r0a_pred["decision_date"].strftime("%Y-%m-%d"), r0a_src["decision_date"].strftime("%Y-%m-%d")),
    ("Latest published reporting month (T - 2)", r0a_pred["origin_period"].strftime("%Y-%m"), r0a_src["origin_period"].strftime("%Y-%m")),
    ("Target month (T)", r0a_pred["target_period"].strftime("%Y-%m"), r0a_src["target_period"].strftime("%Y-%m")),
    ("Actual Type 1 attendance", int(r0a_pred["actual_type1_attendances"]), int(r0a_src["target_type1_attendances"])),
    ("Latest-published prediction (T - 2)", int(r0a_pred["pred_latest_published"]), int(r0a_src["type1_origin"])),
    ("Latest-published absolute error", int(r0a_pred["abs_error_latest_published"]), abs(int(r0a_src["type1_origin"]) - int(r0a_src["target_type1_attendances"]))),
    ("Seasonal naive prediction (T - 12)", int(r0a_pred["pred_seasonal_naive"]), int(r0a_src["type1_target_lag_12"])),
    ("Seasonal naive absolute error", int(r0a_pred["abs_error_seasonal_naive"]), abs(int(r0a_src["type1_target_lag_12"]) - int(r0a_src["target_type1_attendances"]))),
], columns=["Field", "Baseline Output", "Source Feature Dataset"])
display(verify_table)


## 2. Overall 2025 Validation Comparison and Monthly Breakdown

In [ ]:
display(comparison_df[["display_name", "source_column", "n_observations", "n_providers", "mae", "rmse", "wape_pct"]])
display(monthly_df)


## 3. Concise Visualisations

In [ ]:
plt.rcParams.update({"font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), layout="constrained")

labels = ["Latest-published\n(T - 2)", "Seasonal naive\n(T - 12)"]
x = range(len(labels))
width = 0.36
b1 = axes[0].bar([i - width / 2 for i in x], comparison_df["mae"], width=width, color="#245A81", label="MAE")
b2 = axes[0].bar([i + width / 2 for i in x], comparison_df["rmse"], width=width, color="#B56B13", label="RMSE")
axes[0].bar_label(b1, fmt="%.0f", padding=3)
axes[0].bar_label(b2, fmt="%.0f", padding=3)
axes[0].set_xticks(list(x), labels)
axes[0].set_ylabel("Error (Type 1 attendances)")
axes[0].set_title("2025 Validation Overall MAE & RMSE")
axes[0].legend(frameon=False)
axes[0].grid(axis="y", alpha=0.2)

axes[1].plot(monthly_df["target_period"], monthly_df["latest_published_wape_pct"], marker="o", linewidth=2, color="#245A81", label="Latest-published (T - 2)")
axes[1].plot(monthly_df["target_period"], monthly_df["seasonal_naive_wape_pct"], marker="s", linewidth=2, color="#167C80", label="Seasonal naive (T - 12)")
axes[1].set_ylabel("WAPE (%)")
axes[1].set_xlabel("2025 Target Month")
axes[1].set_title("Monthly Validation WAPE (%) Across 2025")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend(frameon=False)
axes[1].grid(axis="y", alpha=0.2)
plt.show()
